# Prodigy_Diagnostics â€” R2.2 (5 esperimenti diagnostici)

Esperimenti diagnostici per CAPIRE Prodigy, non trovare il best.

**Setup comune**: BASELINE_BPTT_864p_PRE_EVENTPROP (CF_FSNN_Net 864p, lambda_sr=0.5).
- 10 ep Ã— 100 step = 1000 step totali
- highway-only, cache F2 (`cache_1500_highway_cut0.0_ou0.0.pt`)
- AdamW non testato qui (Ã¨ solo Prodigy isolation)
- single-seed (fase qualitativa)

**5 esperimenti** (varia 1 lever alla volta):
| ID | Setup | Domanda diagnostica |
|---|---|---|
| P-D1 | lr=1.0 d=1.0 sched=none safeguard=OFF | Esplode o frozen senza safeguard? |
| P-D2 | lr=1.0 d=1.0 sched=none safeguard=ON | Canonical paper. d cresce? |
| P-D3 | lr=1.0 d=1.0 sched=cosine safeguard=ON | Pattern 'scala' emerge? |
| P-D4 | lr=0.1 d=1.0 sched=none safeguard=ON | Replica T30, conferma d quasi-piatto |
| P-D5 | lr=1.0 d=0.5 sched=none safeguard=ON | Brake medio: d cresce piu lento |

**Output per ogni run**: training_log.csv per-epoca + training_batch_log.csv per-batch (1000 righe).
Plot finale: d(step), lr_eff(step), loss(step) per i 5 esperimenti sovrapposti.

**Reference**: `document/PRODIGY_DEEP_STUDY.md` parte 1 (math + source code walkthrough).

In [ ]:
# Cell 1 -- Bootstrap + ENV check
import sys, os, subprocess

# Verifica file critici nel repo root
for f in ['train.py', 'core/network.py', 'config.py',
          'data/cache_1500_highway_cut0.0_ou0.0.pt']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# prodigyopt installed
from prodigyopt import Prodigy
import inspect
sig = inspect.signature(Prodigy.__init__)
print(f'\nProdigy params: {list(sig.parameters.keys())}')
for p in ['safeguard_warmup', 'growth_rate', 'd_coef', 'd0']:
    assert p in sig.parameters, f'MISSING Prodigy param: {p}'
print('[OK] Prodigy v1.1.2 API check')

# train.py supporta i nuovi CLI flags
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--prodigy_safeguard_warmup', '--prodigy_growth_rate', '--prodigy_d_coef']:
    assert flag in help_txt, f'MISSING train.py flag: {flag}'
    print(f'  [OK] train.py supports {flag}')

print('\nENV check passed.')


In [ ]:
# Cell 2 -- EXPERIMENTS list (5 diagnostic configs)
EXPERIMENTS = [
    ('P-D1', 'lr=1.0 d_coef=1.0 sched=none safeguard=OFF', {'lr': 1.0, 'prodigy_d_coef': 1.0, 'scheduler': 'none', 'prodigy_safeguard_warmup': 0, 'prodigy_growth_rate': 'inf'}, "d_hat schizza al primo step (s denom~0). Predizione: d esplode oppure rimane frozen a d0. CONFRONTO con P-D2 isola l'effetto safeguard."),
    ('P-D2', 'lr=1.0 d_coef=1.0 sched=none safeguard=ON', {'lr': 1.0, 'prodigy_d_coef': 1.0, 'scheduler': 'none', 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf'}, 'Config canonical paper. d cresce lentamente, plateau atteso a ~1e-2 entro 10ep (sotto regime ottimo che richiede 100ep+).'),
    ('P-D3', 'lr=1.0 d_coef=1.0 sched=cosine safeguard=ON', {'lr': 1.0, 'prodigy_d_coef': 1.0, 'scheduler': 'cosine', 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf'}, 'Cosine annealing su lr modula dlr = d*lr. Pattern atteso: d cresce, lr decresce -> lr_eff bell-shape.'),
    ('P-D4', 'lr=0.1 d_coef=1.0 sched=none safeguard=ON (config nostra)', {'lr': 0.1, 'prodigy_d_coef': 1.0, 'scheduler': 'none', 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf'}, 'Replica nostra config T30. d cresce ma plateau basso (~1e-3 a 10ep) -> equivalente SGD con lr piccolo.'),
    ('P-D5', 'lr=1.0 d_coef=0.5 sched=none safeguard=ON (brake medio)', {'lr': 1.0, 'prodigy_d_coef': 0.5, 'scheduler': 'none', 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf'}, 'Brake medio: d_hat scalato 0.5x, crescita piu lenta vs P-D2. Stable ma sotto.'),
]

print(f'{len(EXPERIMENTS)} diagnostic experiments configured.')
for exp_id, label, cfg, expected in EXPERIMENTS:
    print(f'\n[{exp_id}] {label}')
    print(f'  config:   {cfg}')
    print(f'  expected: {expected[:100]}...')


In [ ]:
# Cell 3 -- RUN sweep (5 esperimenti, ~10-15 min ciascuno)
import subprocess, sys, time, os, shutil

SKIP_IF_EXISTS = True   # Se True, salta esperimenti gia eseguiti (per resume safe)
RESULTS_DIR = 'results_R2'
os.makedirs(RESULTS_DIR, exist_ok=True)

def _build_cli(exp_id, cfg):
    """Costruisce CLI per train.py da config Prodigy."""
    args = [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', '10', '--max_steps_per_epoch', '100',
        '--batch_size', '8', '--val_batch_size', '32', '--seq_len', '50',
        '--cf_hidden_size', '32', '--cf_rank', '8',
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', '0.5',
        '--scenario_mix', 'highway', '--cut_in_ratio', '0.0',
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--optimizer', 'prodigy',
        '--data_cache', 'data/cache_1500_highway_cut0.0_ou0.0.pt',
        '--lr', str(cfg['lr']),
        '--max_lr', str(cfg['lr']),
        '--scheduler', cfg['scheduler'],
        '--prodigy_d_coef', str(cfg['prodigy_d_coef']),
        '--prodigy_safeguard_warmup', str(cfg['prodigy_safeguard_warmup']),
        '--prodigy_growth_rate', str(cfg['prodigy_growth_rate']),
        '--tag', f'R2_{exp_id}']
    return args

run_results = []
for i, (exp_id, label, cfg, expected) in enumerate(EXPERIMENTS, 1):
    tag = f'R2_{exp_id}'
    src_log = f'checkpoints/{tag}/training_log.csv'
    dst_dir = f'{RESULTS_DIR}/{tag}'
    if SKIP_IF_EXISTS and os.path.isfile(f'{dst_dir}/training_log.csv'):
        print(f'\n[{i}/{len(EXPERIMENTS)}] [SKIP_EXIST] {tag}')
        run_results.append((exp_id, 'skipped'))
        continue
    print(f'\n{'='*70}\n[{i}/{len(EXPERIMENTS)}] {tag}: {label}\n{'='*70}')
    t0 = time.time()
    r = subprocess.run(_build_cli(exp_id, cfg), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    # Sposta checkpoints/<tag>/ -> results_R2/<tag>/
    src_ck = f'checkpoints/{tag}'
    if os.path.isdir(src_ck):
        if os.path.isdir(dst_dir): shutil.rmtree(dst_dir)
        shutil.move(src_ck, dst_dir)
    print(f'\n[{i}/{len(EXPERIMENTS)}] {tag} -> {status} ({el/60:.1f} min)')
    run_results.append((exp_id, status))

print(f'\n--- SWEEP DONE ---')
for exp_id, status in run_results:
    print(f'  {exp_id}: {status}')


In [ ]:
# Cell 4 -- ANALISI per-batch dynamics: d(step), lr_eff(step), loss(step)
import pandas as pd, matplotlib.pyplot as plt, os
import numpy as np

RESULTS_DIR = 'results_R2'
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

summary_rows = []
for exp_id, label, cfg, expected in EXPERIMENTS:
    tag = f'R2_{exp_id}'
    batch_log = f'{RESULTS_DIR}/{tag}/training_batch_log.csv'
    if not os.path.isfile(batch_log):
        print(f'[SKIP plot] {tag}: batch log missing')
        continue
    df = pd.read_csv(batch_log)
    step = df.index.values   # batch step counter
    # d, lr_eff, loss
    if 'prodigy_d' in df.columns:
        axes[0].plot(step, df['prodigy_d'], label=f'{exp_id}: {label[:35]}', alpha=0.8)
        axes[1].plot(step, df['prodigy_lr_eff'], label=exp_id, alpha=0.8)
    if 'loss' in df.columns:
        loss_col = 'loss'
    elif 'total' in df.columns:
        loss_col = 'total'
    else:
        loss_col = df.select_dtypes(include='number').columns[0]
    axes[2].plot(step, df[loss_col], label=exp_id, alpha=0.7)

    # Summary
    d_final = float(df['prodigy_d'].iloc[-1]) if 'prodigy_d' in df.columns else float('nan')
    d_max = float(df['prodigy_d'].max()) if 'prodigy_d' in df.columns else float('nan')
    lr_eff_max = float(df['prodigy_lr_eff'].max()) if 'prodigy_lr_eff' in df.columns else float('nan')
    loss_final = float(df[loss_col].iloc[-1])
    summary_rows.append({'exp': exp_id, 'd_final': d_final, 'd_max': d_max,
                        'lr_eff_max': lr_eff_max, 'loss_final': loss_final,
                        'expected': expected[:80]})

axes[0].set_yscale('log')
axes[0].set_ylabel('prodigy_d (log scale)')
axes[0].set_title('Dinamica adattatore d per batch')
axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8, loc='best')
axes[1].set_yscale('log')
axes[1].set_ylabel('lr_eff = d * lr (log scale)')
axes[1].set_title('Learning rate effettivo per batch')
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8, loc='best')
axes[2].set_ylabel('train loss')
axes[2].set_xlabel('batch step')
axes[2].set_title('Train loss per batch')
axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8, loc='best')
fig.suptitle('R2.2 Prodigy diagnostic experiments (5 setup, per-batch dynamics)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
out = 'opt_plots/R2_prodigy_diagnostics.png'
os.makedirs('opt_plots', exist_ok=True)
fig.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'[saved] {out}')

summary_df = pd.DataFrame(summary_rows)
from IPython.display import display
display(summary_df)


In [ ]:
# Cell 5 -- val_total per epoca (5 esperimenti)
import pandas as pd, matplotlib.pyplot as plt, os

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
for exp_id, label, cfg, expected in EXPERIMENTS:
    log = f'{RESULTS_DIR}/R2_{exp_id}/training_log.csv'
    if not os.path.isfile(log): continue
    df = pd.read_csv(log)
    if 'val_total' in df.columns:
        ax.plot(df['epoch'], df['val_total'], 'o-', label=f'{exp_id}: {label[:40]}', linewidth=1.6)
ax.set_xlabel('epoch'); ax.set_ylabel('val_total')
ax.set_title('R2.2 -- val_total per epoca (5 setup Prodigy)')
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig('opt_plots/R2_val_total_per_epoch.png', dpi=120, bbox_inches='tight')
plt.show()
print('[saved] opt_plots/R2_val_total_per_epoch.png')
